In [ ]:
from src.model import build_credit_model

# Initialize the model
my_model = build_credit_model(X_market, spreads)

# Run the MCMC sampler
with my_model:
    trace = pm.sample(1000, tune=1000, target_accept=0.9)

import matplotlib.pyplot as plt

# 1. Setup Plot Style
plt.style.use('ggplot') 
fig, ax = plt.subplots(figsize=(12, 6))

# 2. Generate Prediction Range
x_plot = np.linspace(0.5, 12, 200)
X_plot = get_bspline_matrix(x_plot, knots)
post_beta = trace_val.posterior["beta"].stack(sample=("chain", "draw")).values
curve_samples = np.dot(X_plot, post_beta)

# 3. Plot the "Uncertainty Cloud" (The Wow Factor)
ax.fill_between(x_plot, 
                np.percentile(curve_samples, 2.5, axis=1), 
                np.percentile(curve_samples, 97.5, axis=1), 
                color='skyblue', alpha=0.3, label="95% Credible Interval")

# 4. Plot the Mean Line
ax.plot(x_plot, curve_samples.mean(axis=1), color='navy', lw=2, label="Bayesian Mean Curve")

# 5. Plot the Market Points
ax.scatter(tenors, spreads, color='crimson', s=100, edgecolors='white', zorder=5, label="Market Quotes")

# 6. Formatting for AdComs
ax.set_title("Malaysian Corporate Credit Curve (Bayesian P-Spline)", fontsize=15, pad=20)
ax.set_xlabel("Tenor (Years)")
ax.set_ylabel("Spread (bps)")
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()